# 重写 Conv2dTransposeReshapeConcat 为 Conv2dReshapeConcatTranspose

In [1]:
import set_env
from d2py.utils.file import mkdir
root_dir = ".temp"
mkdir(f"{root_dir}/logs")

In [2]:
import numpy as np
import onnx
import tvm
from tvm import relay

In [3]:
import torch
from torch.nn import functional as F
from torch import nn
from torch.onnx import OperatorExportTypes, utils

class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, 1, 1, 0, bias=False, groups=1)
        self.conv0 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)
        self.resize_2 = nn.Conv2d(64, 64, 1, 2, 0, bias=False, groups=1)
        self.conv1 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)
        self.resize_4 = nn.Conv2d(64, 64, 1, 4, 0, bias=False, groups=1)
        self.conv2 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)

    def forward(self, x):
        _x = self.conv(x)
        x0 = self.conv0(_x).permute(0, 2, 3, 1) # NCHW => NHWC
        x1 = self.conv1(self.resize_2(_x)).permute(0, 2, 3, 1)
        x2 = self.conv2(self.resize_4(_x)).permute(0, 2, 3, 1)
        x0 = x0.reshape(1, -1, 3)
        x1 = x1.reshape(1, -1, 3)
        x2 = x2.reshape(1, -1, 3)
        x = torch.concat((x0, x1, x2), dim=1)
        return torch.softmax(x, dim=2)

model = M()
model.eval()

shape = 1, 3, 48, 80
input_name = "data"
dtype = "float32"
data_np = np.random.rand(*shape).astype(dtype)
output_name = "test"
xx = torch.rand(*shape, dtype=torch.float32, requires_grad=False)
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=17,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)

onnx_model = onnx.load(f"{root_dir}/{output_name}.onnx")
mod, params = relay.frontend.from_onnx(onnx_model, {input_name: shape}, freeze_params=True)
mod = relay.transform.InferType()(mod)
mod.show()

Exported graph: graph(%data : Float(1, 3, 48, 80, strides=[11520, 3840, 80, 1], requires_grad=0, device=cpu),
      %conv.weight : Float(64, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=1, device=cpu),
      %conv0.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %resize_2.weight : Float(64, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %conv1.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %resize_4.weight : Float(64, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %conv2.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu)):
  %/conv/Conv_output_0 : Float(1, 64, 48, 80, strides=[245760, 3840, 80, 1], requires_grad=0, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1], onnx_name="/conv/Conv"](%data, %conv.weight), scope: __main__.M::/torch.nn.modules.conv.Conv2d::conv # /media/pc/d

In [4]:
print(mod["main"])

fn (%data: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=/conv/Conv.data:0:0 */) -> Tensor[(1, 20160, 3), float32] {
  %0 = nn.conv2d(%data, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 1, 1), float32] span=/conv/Conv.conv.weight:0:0 */, padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 48, 80), float32] span=/conv/Conv:0:0 */;
  %1 = nn.conv2d(%0, meta[relay.Constant][1] /* ty=Tensor[(12, 64, 1, 1), float32] span=/conv0/Conv.conv0.weight:0:0 */, padding=[0, 0, 0, 0], channels=12, kernel_size=[1, 1]) /* ty=Tensor[(1, 12, 48, 80), float32] span=/conv0/Conv:0:0 */;
  %2 = transpose(%1, axes=[0, 2, 3, 1]) /* ty=Tensor[(1, 48, 80, 12), float32] span=/Transpose:0:0 */;
  %3 = nn.conv2d(%0, meta[relay.Constant][2] /* ty=Tensor[(64, 64, 1, 1), float32] span=/resize_2/Conv.resize_2.weight:0:0 */, strides=[2, 2], padding=[0, 0, 0, 0], channels=64, kernel_size=[1, 1]) /* ty=Tensor[(1, 64, 24, 40), float32] span=/resize_2/Conv:0:0 */;
  %4 = 

注册算子：

In [5]:
from tvm.relay.testing import run_infer_type
from tvm.relay.dataflow_pattern import (
    wildcard, is_op, is_tuple,
    # FunctionPattern,
    DFPatternCallback,
    rewrite
)
import tvm
from tvm.ir.attrs import DictAttrs
from tvm import relay, te, topi
from tvm.relay.op import op as _op
from tvm.target import generic_func

@generic_func
def schedule_special_op(attrs, outs, target):
    with target:
        outs = [outs] if isinstance(outs, te.tensor.Tensor) else outs
        output = outs[0]
        sch = te.create_schedule(output.op)   
        return sch

In [6]:
def custom_yolo_class_predict_rel(arg_types, attrs):
    assert len(arg_types) == 3, "type relation arg number mismatch!"
    assert isinstance(attrs, DictAttrs)
    input_shapes = [input_type.shape for input_type in arg_types]
    n0, c0, h0, w0 = input_shapes[0]
    n1, c1, h1, w1 = input_shapes[1]
    n2, c2, h2, w2 = input_shapes[2]

    group = attrs.group
    assert n0 == n1 == n2 and c0 == c1 == c2
    assert c0 % group == 0
    shape = (
        n0, 
        (h0*w0*c0+h1*w1*c1+h2*w2*c2)//group,
        group
    )
    return relay.TensorType(shape, arg_types[0].dtype)

In [7]:
op_name = "vta_special.vta_yolo_class_predict"
_op.register(op_name, r"code(cal yolo_class_predict.)code")
_op.get(op_name).set_num_inputs(3)
_op.get(op_name).add_argument("data1", "Tensor", "The input data tensor.")
_op.get(op_name).add_argument("data2", "Tensor", "The input data tensor.")
_op.get(op_name).add_argument("data3", "Tensor", "The input data tensor.")
_op.get(op_name).set_attrs_type_key("DictAttrs")
_op.get(op_name).add_type_rel(op_name, custom_yolo_class_predict_rel)
_op.get(op_name).set_support_level(10)
_op.register_pattern(op_name, _op.OpPattern.COMM_REDUCE)
_op.register_stateful(op_name, False) # 无状态算子

In [8]:
def yolo_class_predict(x1, x2, x3, group=3):
    attrs = tvm.ir.make_node("DictAttrs", group=group)
    return relay.Call(_op.get(op_name), [x1, x2, x3], attrs=attrs)

In [9]:
x1 = relay.var("x1", relay.TensorType((1, 12, 12, 20), "float32"))
x2 = relay.var("x2", relay.TensorType((1, 12, 24, 40), "float32"))
x3 = relay.var("x3", relay.TensorType((1, 12, 48, 80), "float32"))
y = yolo_class_predict(x1, x2, x3, group=3)
f = relay.Function([x1, x2, x3], y)
f = run_infer_type(f)

In [10]:
print(f)

fn (%x1: Tensor[(1, 12, 12, 20), float32] /* ty=Tensor[(1, 12, 12, 20), float32] */, %x2: Tensor[(1, 12, 24, 40), float32] /* ty=Tensor[(1, 12, 24, 40), float32] */, %x3: Tensor[(1, 12, 48, 80), float32] /* ty=Tensor[(1, 12, 48, 80), float32] */) -> Tensor[(1, 20160, 3), float32] {
  vta_special.vta_yolo_class_predict(%x1, %x2, %x3, __dict__={"group"=3}) /* ty=Tensor[(1, 20160, 3), float32] */
} /* ty=fn (Tensor[(1, 12, 12, 20), float32], Tensor[(1, 12, 24, 40), float32], Tensor[(1, 12, 48, 80), float32]) -> Tensor[(1, 20160, 3), float32] */


In [11]:
@_op.register_compute(op_name)
def output_softmax_transpose_reshape2d_compute(attrs, inputs, out_type):
    """reshape4d_softmax_reshape2d Relay 计算"""
    assert len(inputs) == 3, "输入参数数量不为 3"
    x1, x2, x3 = inputs

    x1 = topi.transpose(x1, axes=[0, 2, 3, 1])
    n, h, w, c = x1.shape
    x1 = topi.reshape(x1, newshape=[n, h * w * c//attrs.group, attrs.group])

    x2 = topi.transpose(x2, axes=[0, 2, 3, 1])
    n, h, w, c = x2.shape
    x2 = topi.reshape(x2, newshape=[n, h * w * c//attrs.group, attrs.group])

    x3 = topi.transpose(x3, axes=[0, 2, 3, 1])
    n, h, w, c = x3.shape
    x3 = topi.reshape(x3, newshape=[n, h * w * c//attrs.group, attrs.group])

    x = topi.concatenate([x1, x2, x3], axis=1)
    x = topi.nn.softmax(x, axis=2)
    return [x]

_op.register_schedule(op_name, schedule_special_op) # 定义调度

GenericFunc(0x9f1fc00)

In [12]:
class Conv2dTransposeReshapeConcatSoftmaxRewrite(DFPatternCallback):
    def __init__(self):
        super().__init__()
        axes = (0, 2, 3, 1)
        newshape = (1, -1, 3)
        self.x0 = wildcard()
        self.x1 = wildcard()
        self.x2 = wildcard()
        self.transpose0 = is_op("transpose")(self.x0).has_attr({"axes": axes})
        self.reshape0 = is_op("reshape")(self.transpose0).has_attr({"newshape": newshape})
        self.transpose1 = is_op("transpose")(self.x1).has_attr({"axes": axes})
        self.reshape1 = is_op("reshape")(self.transpose1).has_attr({"newshape": newshape})
        self.transpose2 = is_op("transpose")(self.x2).has_attr({"axes": axes})
        self.reshape2 = is_op("reshape")(self.transpose2).has_attr({"newshape": newshape})
        self.tuple_op = is_tuple((self.reshape0, self.reshape1, self.reshape2))
        self.cat = is_op("concatenate")(self.tuple_op).has_attr({"axis": 1})
        self.softmax = is_op("nn.softmax")(self.cat)
        self.pattern = self.softmax

    def callback(self, pre, post, node_map):
        x0 = node_map[self.x0][0]
        x1 = node_map[self.x1][0]
        x2 = node_map[self.x2][0]
        reshape2 = node_map[self.reshape2][0]
        shape = relay.transform.InferTypeLocal(reshape2).shape
        return yolo_class_predict(x0, x1, x2, group=shape[-1])

In [13]:
expr = mod["main"]
expr = rewrite(Conv2dTransposeReshapeConcatSoftmaxRewrite(), expr)
run_mod = tvm.IRModule.from_expr(expr)
run_mod = relay.transform.InferType()(run_mod)
run_mod.show()

数值一致性：

In [14]:
data = np.random.normal(0, 1, size=shape).astype("float32")

target = 'llvm'
dev = tvm.device(target, 0)


# 原始模型
with tvm.transform.PassContext(opt_level=3):
    lib = relay.build(mod, target, params=params)
func = lib[lib.libmod_name]
module = tvm.contrib.graph_executor.GraphModule(func(dev))
module.run(**{input_name: data})
output1 = module.get_output(0).numpy()

# 重写后的模型
with tvm.transform.PassContext(opt_level=3):
    lib = relay.build(run_mod, target, params=params)
func = lib[lib.libmod_name]
module = tvm.contrib.graph_executor.GraphModule(func(dev))
module.run(**{input_name: data})
output2 = module.get_output(0).numpy()
np.testing.assert_allclose(output1, output2)

One or more operators have not been tuned. Please tune your model for better performance. Use DEBUG logging level to see more details.


## 变形

In [15]:
import torch
from torch.nn import functional as F
from torch import nn
from torch.onnx import OperatorExportTypes, utils

class M2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, 1, 1, 0, bias=False, groups=1)
        self.conv0 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)
        self.resize_2 = nn.Conv2d(64, 64, 1, 2, 0, bias=False, groups=1)
        self.conv1 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)
        self.resize_4 = nn.Conv2d(64, 64, 1, 4, 0, bias=False, groups=1)
        self.conv2 = nn.Conv2d(64, 12, 1, 1, 0, bias=False, groups=1)

    def forward(self, x):
        _x = self.conv(x)
        x0 = self.conv0(_x).permute(0, 2, 3, 1) # NCHW => NHWC
        x1 = self.conv1(self.resize_2(_x)).permute(0, 2, 3, 1)
        x2 = self.conv2(self.resize_4(_x)).permute(0, 2, 3, 1)
        x0 = x0.flatten(1)
        x1 = x1.flatten(1)
        x2 = x2.flatten(1)
        x = torch.concat((x0, x1, x2), dim=-1)
        x = torch.reshape(x, (1, -1, 3))
        return torch.softmax(x, dim=2)

model = M2()
model.eval()

output_name = "test2"
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=17,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)

onnx_model = onnx.load(f"{root_dir}/{output_name}.onnx")
mod, params = relay.frontend.from_onnx(onnx_model, {input_name: shape}, freeze_params=True)
mod = relay.transform.InferType()(mod)
mod.show()

Exported graph: graph(%data : Float(1, 3, 48, 80, strides=[11520, 3840, 80, 1], requires_grad=0, device=cpu),
      %conv.weight : Float(64, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=1, device=cpu),
      %conv0.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %resize_2.weight : Float(64, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %conv1.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %resize_4.weight : Float(64, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu),
      %conv2.weight : Float(12, 64, 1, 1, strides=[64, 1, 1, 1], requires_grad=1, device=cpu)):
  %/conv/Conv_output_0 : Float(1, 64, 48, 80, strides=[245760, 3840, 80, 1], requires_grad=0, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1], onnx_name="/conv/Conv"](%data, %conv.weight), scope: __main__.M2::/torch.nn.modules.conv.Conv2d::conv # /media/pc/

In [16]:
# 原始模型
with tvm.transform.PassContext(opt_level=3):
    lib = relay.build(mod, target, params=params)
func = lib[lib.libmod_name]
module = tvm.contrib.graph_executor.GraphModule(func(dev))
module.run(**{input_name: data})
output3 = module.get_output(0).numpy()

In [17]:
class Conv2dTransposeFlattenConcatReshapeSoftmaxRewrite(DFPatternCallback):
    def __init__(self):
        super().__init__()
        axes = (0, 2, 3, 1)
        self.x0 = wildcard()
        self.x1 = wildcard()
        self.x2 = wildcard()
        self.transpose0 = is_op("transpose")(self.x0).has_attr({"axes": axes})
        self.flatten0 = is_op("nn.batch_flatten")(self.transpose0)
        self.transpose1 = is_op("transpose")(self.x1).has_attr({"axes": axes})
        self.flatten1 = is_op("nn.batch_flatten")(self.transpose1)
        self.transpose2 = is_op("transpose")(self.x2).has_attr({"axes": axes})
        self.flatten2 = is_op("nn.batch_flatten")(self.transpose2)
        self.tuple_op = is_tuple((self.flatten0, self.flatten1, self.flatten2))
        self.cat = is_op("concatenate")(self.tuple_op)
        self.reshape = is_op("reshape")(self.cat)
        self.softmax = is_op("nn.softmax")(self.reshape)
        self.pattern = self.softmax

    def callback(self, pre, post, node_map):
        x0 = node_map[self.x0][0]
        x1 = node_map[self.x1][0]
        x2 = node_map[self.x2][0]
        reshape = node_map[self.reshape][0]
        shape = relay.transform.InferTypeLocal(reshape).shape
        return yolo_class_predict(x0, x1, x2, group=shape[-1])

In [18]:
expr = mod["main"]
expr = rewrite(Conv2dTransposeFlattenConcatReshapeSoftmaxRewrite(), expr)
run_mod = tvm.IRModule.from_expr(expr)
run_mod = relay.transform.InferType()(run_mod)
run_mod.show()

In [19]:
data = np.random.normal(0, 1, size=shape).astype("float32")

target = 'llvm'
dev = tvm.device(target, 0)


# 原始模型
with tvm.transform.PassContext(opt_level=3):
    lib = relay.build(mod, target, params=params)
func = lib[lib.libmod_name]
module = tvm.contrib.graph_executor.GraphModule(func(dev))
module.run(**{input_name: data})
output1 = module.get_output(0).numpy()

# 重写后的模型
with tvm.transform.PassContext(opt_level=3):
    lib = relay.build(run_mod, target, params=params)
func = lib[lib.libmod_name]
module = tvm.contrib.graph_executor.GraphModule(func(dev))
module.run(**{input_name: data})
output2 = module.get_output(0).numpy()
np.testing.assert_allclose(output1, output2)